# 1. Pytorch加载数据

① Pytorch中加载数据需要Dataset、Dataloader。

 - Dataset提供一种方式去获取每个数据及其对应的label，告诉我们总共有多少个数据。
 - Dataloader为后面的网络提供不同的数据形式，它将一批一批数据进行一个打包。

# 2. 常用数据集两种形式

# ① 常用的第一种数据形式，文件夹的名称是它的label。
  #   train/
  # ants/
  #   001.jpg
  #   002.jpg
  # bees/
  #   101.jpg
  #   102.jpg

# ② 常用的第二种形式，label为文本格式。文本名称为图片名称，文本中的内容为对应的label。比如某张图片是111.jpg，那么就有一个111.txt的文本文件，文本内容是它的label。

In [6]:
from torch.utils.data import Dataset # 从 torch.utils.data 这个模块里，把名为 Dataset 的对象（类）导入到当前文件/Notebook 的命名空间中，之后就可以直接用 Dataset 这个名字，而不用写全路径。
help(Dataset)

Help on class Dataset in module torch.utils.data.dataset:

class Dataset(typing.Generic)
 |  An abstract class representing a :class:`Dataset`.
 |  
 |  All datasets that represent a map from keys to data samples should subclass
 |  it. All subclasses should overwrite :meth:`__getitem__`, supporting fetching a
 |  data sample for a given key. Subclasses could also optionally overwrite
 |  :meth:`__len__`, which is expected to return the size of the dataset by many
 |  :class:`~torch.utils.data.Sampler` implementations and the default options
 |  of :class:`~torch.utils.data.DataLoader`.
 |  
 |  .. note::
 |    :class:`~torch.utils.data.DataLoader` by default constructs a index
 |    sampler that yields integral indices.  To make it work with a map-style
 |    dataset with non-integral indices/keys, a custom sampler must be provided.
 |  
 |  Method resolution order:
 |      Dataset
 |      typing.Generic
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __add__(self, oth

# 3. 路径直接加载数据

In [ ]:
from PIL import Image # 第三方图像处理库 Pillow，提供Image类

img_path = "Data/FirstTypeData/train/ants/0013035.jpg"        
img = Image.open(img_path)  # 返回对象是 PIL.Image.Image 类的实例
img.show()

# 4. Dataset加载数据

In [ ]:
"""
MyData 的作用是把你磁盘上“按某种规则存放的原始数据”（路径、读图、标签规则、预处理）
封装成 PyTorch 统一可迭代的 Dataset（实现 __len__ 和 __getitem__），
从而能被 DataLoader 按索引取样、组 batch。

何时不需要用它：当你的数据格式已经被现成数据集/通用加载器覆盖时（例如文件夹名即类别时直接用 torchvision.datasets.ImageFolder，
或直接用内置数据集如 CIFAR10/MNIST），就不必自己写自定义 Dataset。
"""

from torch.utils.data import Dataset
from PIL import Image
import os

# 自定义数据集类，并继承 Dataset 类。
# Dataset 是一个抽象类，里面定义了一些方法，但没有具体实现。
# 我们在 MyData 类中实现这些方法，就可以创建一个可用的数据集了。
class MyData(Dataset):     
    def __init__(self,root_dir,label_dir):    # 当创建一个事例对象时，会自动调用该函数。即构造函数。
        self.root_dir = root_dir # self.root_dir 相当于类中的全局变量
        self.label_dir = label_dir     
        self.path = os.path.join(self.root_dir,self.label_dir) # 字符串拼接，根据是Windows或Lixus系统情况进行拼接   # 例：Data/FirstTypeData/train + ants → Data/FirstTypeData/train/ants             
        self.img_path = os.listdir(self.path) # 获得路径下所有图片的名字列表。例如：['001.jpg', '002.jpg', ...]。
        
    '''
    定义取该对象的下标时,所处理的过程
    例如：当 idx=0 时，取到第一个样本；当 idx=1 时，取到第二个样本；以此类推。
    '''
    def __getitem__(self,idx):
        img_name = self.img_path[idx]   # 从文件名列表中取出第 idx 个文件的名字。例如：当 idx=0 时，img_name 就是 '001.jpg'。
        img_item_path = os.path.join(self.root_dir,self.label_dir,img_name)  # 根据路径和文件名拼接成一个完整的路径。例如：Data/FirstTypeData/train + ants + 001.jpg → Data/FirstTypeData/train/ants/001.jpg   
        img = Image.open(img_item_path) # 返回的数据类型 PIL.Image.Image
        label = self.label_dir
        return img, label
    
    def __len__(self):
        return len(self.img_path)
    
root_dir = "Data/FirstTypeData/train"
ants_label_dir = "ants"
bees_label_dir = "bees"
ants_dataset = MyData(root_dir, ants_label_dir)
bees_dataset = MyData(root_dir, bees_label_dir)
print(len(ants_dataset))
print(len(bees_dataset))
train_dataset = ants_dataset + bees_dataset # train_dataset 就是两个数据集的集合了     
print(len(train_dataset))

img,label = train_dataset[200]
print("label：",label)
img.show()

124
121
245
label： bees
